# JAX, but start where NumPy ends

DevFest Kisumu 2026 · beginner to intermediate

Have you ever copied a NumPy function, swapped in `jnp`, and watched it fall over?

That is the talk.

One small function. Same code. CPU, GPU, or TPU.

## On Kaggle
Settings → Accelerator → TPU. Internet ON. Run from the top.
Works on CPU for rehearsal. TPU is the point on the day.

## 0. Setup
Confirm the runtime sees a device. On a Kaggle TPU you want more than a CPU.

In [ ]:
import jax
import jax.numpy as jnp
from jax import random

print("jax", jax.__version__)
print("devices:", jax.devices())
print("device_count:", jax.device_count())
print("default backend:", jax.default_backend())


## 1. Mental model

JAX is not another PyTorch. It is a transform engine.

- `jnp` looks like NumPy. Arrays do not mutate.
- Transforms (`jit`, `grad`, `vmap`) wrap pure functions. They compose.
- XLA traces the function, fuses ops, and targets your hardware.
- First call is the compile tax. Second call is the point.

## 2. Where NumPy ends
Start with code you already know. Then swap.

In [ ]:
import numpy as np

# this is numpy

def selu_np(x, alpha=1.67, scale=1.05):
    return scale * np.where(x > 0, x, alpha * np.exp(x) - alpha)

x_np = np.arange(5, dtype=np.float32)
print(selu_np(x_np))

In [ ]:
# this is jax numpy (jnp), kinda looks the same

def selu(x, alpha=1.67, scale=1.05):
    return scale * jnp.where(x > 0, x, alpha * jnp.exp(x) - alpha)

x = jnp.arange(5, dtype=jnp.float32)
print(selu(x))


In [ ]:
# this raises. arrays are immutable.
try:
    x[0] = 10
except Exception as e:
    print(type(e).__name__, ":", e)
    

Mutation would make the graph a lie.

In [ ]:
# functional update. new array.
x2 = x.at[0].set(10)
print("original", x)
print("updated ", x2)

In [ ]:
key = random.key(0)

# WRONG — same numbers forever
a = random.normal(key, (3,))
b = random.normal(key, (3,))
print("reused key")
print(a)
print(b)
print("equal?", jnp.allclose(a, b))

same key, same noise. dropout that never drops.

In [ ]:
# RIGHT — split, then consume
key, k1, k2 = random.split(key, 3)
a = random.normal(k1, (3,))
b = random.normal(k2, (3,))
print("split keys")
print(a)
print(b)
print("equal?", jnp.allclose(a, b))

split, then consume. Thread the key. No global RNG(random number generator)

the jax.random package provides a number of routines for deterministic generation of sequences of pseudorandom numbers(https://docs.jax.dev/en/latest/jax.random.html)



## 4. One function. Three transforms.
Stay on `selu`. Do not write a new one.

selu - scaled exponential linear unit activation is an activation function designed to help neural networks train more effectively by keeping the output of each layer automatically normalized. Unlike simpler functions like ReLU, it not only introduces non-linearity but also helps to stabilize the learning process.
(https://www.geeksforgeeks.org/deep-learning/selu-activation-function-in-neural-network/)

In [ ]:
from jax import grad, vmap, jit

dselu = grad(lambda x: selu(x).sum())
print("grad at 0.5:", dselu(0.5))
print("grad at -0.5:", dselu(-0.5))


grad of the same function. Positive side is linear. Negative side is the exp.

In [ ]:
xs = jnp.linspace(-2.0, 2.0, 5)
print("vmap selu:", vmap(selu)(xs))

vmap batches it. Same function. No loop. 
vmap(https://docs.jax.dev/en/latest/_autosummary/jax.vmap.html)


In [ ]:
# compose: jit(grad(vmap(f)))
f = jit(grad(lambda x: vmap(selu)(x).sum()))

print("first call:", f(xs))   # compile tax
print("second call:", f(xs))  # the point

same function. Composed. First call is the tax. Second call is why you are here ...

## 5. Sharp bits that may frustrate sometimes

In [ ]:
# silent float32. SciPy muscle memory wants float64. JAX does not.
print("default dtype:", jnp.arange(3).dtype)
print("1.0 / 3 is", (jnp.array(1.0) / 3).dtype)

In [ ]:
# tracer leak: a changing Python value is captured at trace time.
scale = 1.0

@jit
def scaled(x):
    return x * scale

print("scale=1.0 ->", scaled(jnp.array(2.0)))
scale = 99.0
print("scale=99  ->", scaled(jnp.array(2.0)))  # still 2.0

it captured 1.0. Pass scale as an argument if it must change.

In [ ]:
# dynamic shape. Boolean index is not jit-friendly without padding.
@jit
def take_positive(x):
    return x[x > 0]

try:
    print(take_positive(jnp.array([-1.0, 2.0, -3.0, 4.0])))
except Exception as e:
    print(type(e).__name__, ":", str(e)[:180])

the shape depends on the data. Pad, or do not jit that path.

## 6. When JAX, when not

**Reach for JAX**
- You want `grad` / `vmap` / `scan` to compose.
- TPU, or one codepath from CPU to GPU to TPU.
- Functional style is a feature, not a tax.

**Stay on PyTorch**
- Hugging Face trainers, many SOTA repos.
- You need eager debugging and dynamic graphs first.
- The team has no XLA(accelerated linear algebra) muscle yet.

XLA( accelerated linear algebra: https://github.com/openxla/xla) is an open-source compiler for machine learning that optimizes computation graphs into efficient machine code to improve model execution time for training and inference across CPUs, GPUs, and ML accelerators. It is part of the OpenXLA project, which integrates with frameworks like TensorFlow, PyTorch, and JAX

## 7. Close

Learn in this order.
1. JAX 101: `jnp`, keys, `jit`, `grad`, `vmap`.
2. One loop: `jit(value_and_grad)` + Optax on a tiny MLP.
3. Flax NNX.
4. 201: `scan`, sharding, Orbax.

- https://docs.jax.dev
- https://docs.jaxstack.ai

# Questions ...